In [20]:
pip install yfinance transformers feedparser beautifulsoup4 pandas requests transformers

In [21]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import yfinance as yf
import feedparser

SEC Request Headers

In [22]:
# Required by the SEC to identify the requester.
#This ensures compliant and responsible data access when using SEC's EDGAR API.

HEADERS = {
    "User-Agent": "MarioYanez CaliforniaStateUniversityFresno myanez987@mail.fresnostate.edu",
    "Accept-Encoding": "gzip, deflate"
}

Finbert setup

In [23]:
#Loads a financial domain version for BERT (FinBERT) for classifying text as positive, negative, or neutral.
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


Tickers

In [24]:
ticker_input = input("Enter comma-separated stock tickers (e.g., NVDA, AAPL, MSFT): ")
tickers = [t.strip().upper() for t in ticker_input.split(",") if t.strip()]

search_terms = {t: t for t in tickers}
print(f"Will analyze: {list(search_terms.keys())}")

Finviz Scraper

In [25]:
# Pulling news headlines from Finviz
#Scrapes the latest headlines from the stock's Finviz profile page
def get_finviz_headlines(ticker):
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        news_table = soup.find("table", class_="fullview-news-outer")
        rows = news_table.find_all("tr") if news_table else []
        headlines = [row.a.get_text(strip=True) for row in rows if row.a]
        print(f"Pulled {len(headlines)} Finviz headlines for {ticker}")
        return headlines
    except Exception as e:
        print(f"Finviz fetch failed for {ticker}: {e}")
        return []

Google News RSS

In [26]:
#Getting headlines from Google News RSS
#Queries Google News RSS feed using the format [TICKER] stock

def get_google_news_rss(ticker):
    query = f"{ticker} stock"
    url = f"https://news.google.com/rss/search?q={query.replace(' ', '+')}"
    try:
        feed = feedparser.parse(url)
        entries = [entry.title for entry in feed.entries]
        print(f"Pulled {len(entries)} Google RSS headlines for {ticker}")
        return entries
    except Exception as e:
        print(f"Google RSS fetch failed for {ticker}: {e}")
        return []




SEC

Getting CIK Map

In [27]:
#Pulls public JSON file from the SEC and returns a dictionaty mapping tickers to a 10 digit CIK codes

def get_cik_map():
    url = "https://www.sec.gov/files/company_tickers.json"
    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()
        cik_map = {
            entry["ticker"]: str(entry["cik_str"]).zfill(10)
            for entry in data.values()
        }
        return cik_map
    except Exception as e:
        print(f"Failed to fetch CIK map: {e}")
        return {}
CIK_MAP = get_cik_map()

Get SEC filings

In [28]:
#Gets last SEC 10-Q Filings
#Pulls and parse the four most recent quarterly filings from EDGAR

def get_recent_sec_filing_texts(ticker, form_type="10-Q", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, (f, d) in enumerate(zip(forms, documents))
        if "10-q" in f.lower() or "10-q" in d.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = documents[i]
        link = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{doc_name}"
        print(f"[DEBUG] Trying SEC link: {link}")

        try:
            response = requests.get(link, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text(separator=" ", strip=True)

            if len(text.strip()) < 100:
              continue

            texts.append(text[:1500])
        except Exception as e:
            print(f"Could not parse SEC filing for {ticker}: {e}")

    print(f"{ticker}: Parsed {len(texts)} {form_type} filings")
    return texts


8K

In [29]:
def get_recent_sec_filing_texts(ticker, form_type="8-K", count=2):
    cik = CIK_MAP.get(ticker.upper())
    if not cik:
        print(f"CIK not found for {ticker}")
        return []

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    print(f"[DEBUG] URL: {url}")
    try:
        res = requests.get(url, headers=HEADERS)
        print(f"[DEBUG] Status Code: {res.status_code}")
        res.raise_for_status()
    except Exception as e:
        print(f"Failed to get filings for {ticker}: {e}")
        return []

    data = res.json()
    filings = data.get("filings", {}).get("recent", {})
    forms = filings.get("form", [])
    documents = filings.get("primaryDocument", [])
    accession_numbers = filings.get("accessionNumber", [])

    indices = [
        i for i, f in enumerate(forms)
        if form_type.lower() in f.lower()
    ]

    print(f"[DEBUG] Matching SEC indices: {indices}")
    texts = []

    for i in indices[:count]:
        acc_num = accession_numbers[i].replace("-", "")
        doc_name = documents[i]
        link = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{doc_name}"
        print(f"[DEBUG] Trying SEC link: {link}")

        try:
            response = requests.get(link, headers=HEADERS)
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text(separator=" ", strip=True)

            if len(text.strip()) < 100:
                continue

            texts.append(text[:1500])
        except Exception as e:
            print(f"Could not parse SEC filing for {ticker}: {e}")

    print(f"{ticker}: Parsed {len(texts)} {form_type} filings")
    return texts

Finbert Sentiment Analysis

In [30]:
#Classifies each headline or snippet as poisitive, negative, or neutral
#Converts the result into numneric score for aggregation

def analyze_headlines_with_finbert(headlines):
    sentiments = []
    for headline in headlines:
        try:

            tokens = tokenizer.tokenize(headline)
            if len(tokens) > 512:
                tokens = tokens[:512]
                headline = tokenizer.convert_tokens_to_string(tokens)

            result = finbert(headline)[0]
            label = result["label"].lower()
            score = result["score"]
            print(f"{headline[:60]}... → {label} ({score:.4f})")

            if label == "positive":
                sentiments.append(score)
            elif label == "negative":
                sentiments.append(-score)
            else:
                sentiments.append(0.01)
        except Exception as e:
            print(f"Sentiment error on headline: {headline[:50]} — {e}")
    return sentiments

Main Execution

In [31]:
def run_pipeline():
    summary_data = []
    for ticker, query in search_terms.items():
        print(f"Processing {ticker}")
        finviz_headlines = get_finviz_headlines(ticker)
        google_headlines = get_google_news_rss(ticker)
        sec_texts = get_recent_sec_filing_texts(ticker)
        combined_texts = finviz_headlines + google_headlines + sec_texts
        if combined_texts:
            sentiment_scores = analyze_headlines_with_finbert(combined_texts)
            avg_score = sum(sentiment_scores) / len(sentiment_scores)
            label = (
                "Buy" if avg_score > 0.1 else
                "Hold" if avg_score >= -0.1 else
                "Sell" if avg_score > -0.4 else
                "Strong Sell"
            )
            summary_data.append({
                "Ticker": ticker,
                "Average Score": avg_score,
                "Sentiment Label": label,
            })
        else:
            print(f"No headlines found for {ticker}")
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv("sentiment_regression_data_2025.csv", index=False)
    print("Saved as 'sentiment_regression_data_2025.csv'")
    print(summary_df)
if __name__ == "__main__":
    run_pipeline()